# 초기 fastText 확장 ESG 사전 생성

## 분석 목적

이 노트북의 목적은 seed ESG dictionary를 출발점으로 fastText 임베딩 유사도를 활용해 E/S/G 차원별 확장 사전을 생성하는 것이다.

## 분석 방법

- seed dictionary의 E/S/G 용어를 정규화한다.
- fastText 모델에서 seed 용어와 유사한 후보 단어를 탐색한다.
- threshold별 후보 단어를 E/S/G 차원별로 정리한다.
- 확장 사전과 수동 검토용 후보 목록을 생성한다.


In [8]:
from pathlib import Path
import html
import re
import subprocess
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import fasttext
    from huggingface_hub import hf_hub_download
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "fasttext-wheel", "huggingface_hub"
    ])
    import fasttext
    from huggingface_hub import hf_hub_download

# Colab-friendly Drive mount. Locally, this block is a no-op.
try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

DRIVE_ROOT = Path("/content/drive/MyDrive/UD_26")
LOCAL_ROOT = Path.cwd()

HF_FASTTEXT_REPO_ID = "facebook/fasttext-ko-vectors"
HF_FASTTEXT_FILENAME = "model.bin"
FASTTEXT_TOP_K = 500
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.80, 0.90, 1.00]
REVIEW_ROWS_PER_THRESHOLD_DIMENSION = 20

SEED_DICTIONARY_CANDIDATES = [
    DRIVE_ROOT / "final" / "seed_dictionary.csv",
    DRIVE_ROOT / "data" / "seed_dictionary.csv",
    LOCAL_ROOT / "final" / "seed_dictionary.csv",
    LOCAL_ROOT / "data" / "seed_dictionary.csv",
]


def first_existing_path(candidates, label):
    for path in candidates:
        if path.exists():
            return path
    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {label}. Searched:\n{searched}")


def resolve_output_dir():
    if DRIVE_ROOT.exists():
        return DRIVE_ROOT / "final" / "expanded_dictionaries"
    return LOCAL_ROOT / "final" / "expanded_dictionaries"

SEED_DICTIONARY_PATH = first_existing_path(SEED_DICTIONARY_CANDIDATES, "seed_dictionary.csv")
OUTPUT_DIR = resolve_output_dir()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_CANDIDATES_PATH = OUTPUT_DIR / "fasttext_raw_candidates_all_seeds.csv"

print("SEED_DICTIONARY_PATH:", SEED_DICTIONARY_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RAW_CANDIDATES_PATH:", RAW_CANDIDATES_PATH)
print("HF_FASTTEXT_REPO_ID:", HF_FASTTEXT_REPO_ID)
print("FASTTEXT_TOP_K:", FASTTEXT_TOP_K)
print("THRESHOLDS:", THRESHOLDS)


SEED_DICTIONARY_PATH: /content/drive/MyDrive/UD_26/final/seed_dictionary.csv
OUTPUT_DIR: /content/drive/MyDrive/UD_26/final/expanded_dictionaries
RAW_CANDIDATES_PATH: /content/drive/MyDrive/UD_26/final/expanded_dictionaries/fasttext_raw_candidates_all_seeds.csv
HF_FASTTEXT_REPO_ID: facebook/fasttext-ko-vectors
FASTTEXT_TOP_K: 500
THRESHOLDS: [0.1, 0.2, 0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8, 0.9, 1.0]


In [9]:
seed_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")

required_seed_columns = {"dimension", "seed_term"}
missing_seed = required_seed_columns - set(seed_df.columns)
if missing_seed:
    raise ValueError(f"seed_dictionary.csv missing columns: {sorted(missing_seed)}")

print("seed_df shape:", seed_df.shape)
display(seed_df.head())


seed_df shape: (30, 6)


,dimension,seed_term,pattern,source_basis,source_titles,notes
0,E,탄소,탄소,Bloomberg ESG climate change; Sautner-style cl...,Bloomberg ESG framework; Sautner-style climate...,carbon / climate exposure core seed
1,E,온실가스,온실가스|GHG,Bloomberg ESG climate change; GHG emissions li...,Bloomberg ESG framework; Sautner-style climate...,greenhouse gas emissions seed
2,E,탄소중립,탄소중립,Bloomberg ESG climate change; net-zero transit...,Bloomberg ESG framework; Sautner-style climate...,net-zero transition Korean term
3,E,넷제로,넷제로|net zero|net-zero,Bloomberg ESG climate change; net-zero transit...,Bloomberg ESG framework; Sautner-style climate...,net-zero borrowed term
4,E,재생에너지,재생에너지|renewable energy,Bloomberg ESG water/energy management; climate...,Bloomberg ESG framework; Sautner-style climate...,renewable energy seed


In [10]:
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_term(term):
    return normalize_text(term)


def threshold_label(theta):
    return f"{theta:.2f}".replace(".", "_")


def dictionary_name(theta):
    return f"expanded_dictionary_theta_{threshold_label(theta)}"


def pattern_from_term(term):
    return re.escape(normalize_term(term))

seed_df = seed_df.copy()
seed_df["dimension"] = seed_df["dimension"].map(normalize_term)
seed_df["seed_term"] = seed_df["seed_term"].map(normalize_term)
if "pattern" not in seed_df.columns:
    seed_df["pattern"] = ""

seed_df = seed_df[seed_df["dimension"].ne("") & seed_df["seed_term"].ne("")].copy()

print("normalized seed rows:", len(seed_df))
display(seed_df.head())


normalized seed rows: 30


,dimension,seed_term,pattern,source_basis,source_titles,notes
0,E,탄소,탄소,Bloomberg ESG climate change; Sautner-style cl...,Bloomberg ESG framework; Sautner-style climate...,carbon / climate exposure core seed
1,E,온실가스,온실가스|GHG,Bloomberg ESG climate change; GHG emissions li...,Bloomberg ESG framework; Sautner-style climate...,greenhouse gas emissions seed
2,E,탄소중립,탄소중립,Bloomberg ESG climate change; net-zero transit...,Bloomberg ESG framework; Sautner-style climate...,net-zero transition Korean term
3,E,넷제로,넷제로|net zero|net-zero,Bloomberg ESG climate change; net-zero transit...,Bloomberg ESG framework; Sautner-style climate...,net-zero borrowed term
4,E,재생에너지,재생에너지|renewable energy,Bloomberg ESG water/energy management; climate...,Bloomberg ESG framework; Sautner-style climate...,renewable energy seed


In [11]:
def build_seed_query_frame(seed_source_df):
    rows = []
    for _, row in seed_source_df.iterrows():
        values = [row.get("seed_term", "")]
        pattern = row.get("pattern", "")
        if pd.notna(pattern):
            values.extend(str(pattern).split("|"))

        seen = set()
        for value in values:
            query_term = normalize_term(value)
            if not query_term or query_term.lower() == "nan" or query_term in seen:
                continue
            seen.add(query_term)
            rows.append({
                "dimension": row["dimension"],
                "seed_term": normalize_term(row["seed_term"]),
                "query_term": query_term,
            })

    return pd.DataFrame(rows).drop_duplicates(["dimension", "seed_term", "query_term"]).reset_index(drop=True)


seed_query_df = build_seed_query_frame(seed_df)
print("seed query rows:", len(seed_query_df))
display(seed_query_df.groupby("dimension").size().rename("query_terms"))


seed query rows: 54


,query_terms
dimension,
E,18
G,18
S,18


In [12]:
model_path = hf_hub_download(repo_id=HF_FASTTEXT_REPO_ID, filename=HF_FASTTEXT_FILENAME)
fasttext_model = fasttext.load_model(model_path)

raw_candidate_rows = []
for _, row in seed_query_df.iterrows():
    query_term = row["query_term"]
    try:
        neighbors = fasttext_model.get_nearest_neighbors(query_term, k=FASTTEXT_TOP_K)
    except Exception as exc:
        print(f"Skipping {query_term!r}: {exc}")
        continue

    for similarity, candidate_term in neighbors:
        candidate_term = normalize_term(candidate_term)
        if not candidate_term or candidate_term == query_term:
            continue
        raw_candidate_rows.append({
            "dimension": row["dimension"],
            "seed_term": row["seed_term"],
            "query_term": query_term,
            "candidate_term": candidate_term,
            "similarity": float(similarity),
        })

raw_candidates_df = pd.DataFrame(raw_candidate_rows)
if raw_candidates_df.empty:
    raise ValueError("No fastText candidates were generated. Check seed terms and model loading.")

raw_candidates_df = raw_candidates_df.drop_duplicates(
    ["dimension", "seed_term", "query_term", "candidate_term"]
).sort_values(["dimension", "seed_term", "query_term", "similarity"], ascending=[True, True, True, False])
raw_candidates_df.to_csv(RAW_CANDIDATES_PATH, index=False, encoding="utf-8-sig")

print("raw candidate rows:", len(raw_candidates_df))
print("saved:", RAW_CANDIDATES_PATH)
display(raw_candidates_df.head(20))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.bin:   0%|          | 0.00/7.24G [00:00<?, ?B/s]

raw candidate rows: 26989
saved: /content/drive/MyDrive/UD_26/final/expanded_dictionaries/fasttext_raw_candidates_all_seeds.csv


,dimension,seed_term,query_term,candidate_term,similarity
2500,E,넷제로,net zero,괜찮아ᄏ입술펴이러는거임그래서내가좀벌렷더니혀를넣고ᄌᄅ임나는너무더러워서오빠가자햇더니가슴...,0.714383
2501,E,넷제로,net zero,352831292E5FBBEABEF7C1DFBAD0B7F9BAB02C5FC1B6C1...,0.701920
2502,E,넷제로,net zero,5BB0EDB3ADB5B55D32303131B3E2B4EBBAF12DB0ED312D...,0.696082
2503,E,넷제로,net zero,니집이나빨리가자이러는거임암튼그러케계속감도착한후오빠가소파에앉더니이리와ᄒᄒ이러는거임그래...,0.695077
2504,E,넷제로,net zero,구시가팔리라키익시아린도스이아리소스칼리테아페프코스콜림비아라르도스호텔,0.695062
2505,E,넷제로,net zero,사이트게시판자유게시판유머게시판질답게시판요청게시판플래시게임갤러리스타갤러리모델갤러리얼짱...,0.695015
2506,E,넷제로,net zero,중구울주군세종시경기고양시동두천시연천군광명시광주시구리시군포시김포시남양주시부천시성남시수...,0.694936
2507,E,넷제로,net zero,464B4949B8AEC6F7C6AE2DC0AFBAF1C4F5C5CDBDBABBEA...,0.693545
2508,E,넷제로,net zero,GeometryStoriesAthletesClearance남성자전거산악로드피트니스터...,0.693080
2509,E,넷제로,net zero,피진어통가어투르크멘어투발루어티그리냐어티베트어파나갈로어파슈토어파키스탄어파피아멘투어팔라...,0.692973


In [13]:
def build_seed_dictionary_frame(seed_source_df):
    rows = []
    for _, row in seed_source_df.iterrows():
        values = [row.get("seed_term", "")]
        pattern = row.get("pattern", "")
        if pd.notna(pattern):
            values.extend(str(pattern).split("|"))

        seen = set()
        for value in values:
            term = normalize_term(value)
            if not term or term.lower() == "nan" or term in seen:
                continue
            seen.add(term)
            rows.append({
                "dictionary_name": "seed_dictionary",
                "threshold": np.nan,
                "dimension": row["dimension"],
                "seed_term": normalize_term(row["seed_term"]),
                "candidate_term": term,
                "pattern": pattern_from_term(term),
                "similarity": 1.0,
                "source": "seed",
                "seed_terms_matched": normalize_term(row["seed_term"]),
                "query_terms_matched": term,
                "keep_review": True,
                "exclude_reason": "",
            })

    seed_dictionary = pd.DataFrame(rows)
    return seed_dictionary.drop_duplicates(["dimension", "candidate_term"]).reset_index(drop=True)


def build_expanded_dictionary_for_threshold(theta):
    seed_rows = build_seed_dictionary_frame(seed_df)
    seed_rows["dictionary_name"] = dictionary_name(theta)
    seed_rows["threshold"] = theta

    filtered = raw_candidates_df.loc[raw_candidates_df["similarity"].ge(theta)].copy()

    grouped_rows = []
    group_columns = ["dimension", "candidate_term"]
    for (dimension, candidate_term), group in filtered.groupby(group_columns, sort=False):
        best = group.sort_values("similarity", ascending=False).iloc[0]
        seed_terms = "; ".join(sorted(group["seed_term"].dropna().unique()))
        query_terms = "; ".join(sorted(group["query_term"].dropna().unique()))
        grouped_rows.append({
            "dictionary_name": dictionary_name(theta),
            "threshold": theta,
            "dimension": dimension,
            "seed_term": best["seed_term"],
            "candidate_term": candidate_term,
            "pattern": pattern_from_term(candidate_term),
            "similarity": float(best["similarity"]),
            "source": "fasttext_candidate",
            "seed_terms_matched": seed_terms,
            "query_terms_matched": query_terms,
            "keep_review": "REVIEW",
            "exclude_reason": "",
        })

    candidate_df = pd.DataFrame(grouped_rows)
    out_df = pd.concat([seed_rows, candidate_df], ignore_index=True)
    out_df = out_df.drop_duplicates(["dimension", "candidate_term"], keep="first")
    return out_df.sort_values(
        ["dimension", "source", "similarity", "candidate_term"],
        ascending=[True, True, False, True],
    ).reset_index(drop=True)

seed_dictionary_df = build_seed_dictionary_frame(seed_df)
print("seed dictionary rows:", len(seed_dictionary_df))
display(seed_dictionary_df.groupby("dimension").size().rename("seed_terms"))


seed dictionary rows: 53


,seed_terms
dimension,
E,18
G,17
S,18


In [14]:
seed_dictionary_path = OUTPUT_DIR / "seed_dictionary_normalized_for_comparison.csv"
seed_dictionary_df.to_csv(seed_dictionary_path, index=False, encoding="utf-8-sig")

expanded_dictionary_map = {}
saved_files = [seed_dictionary_path, RAW_CANDIDATES_PATH]
summary_rows = []

for theta in THRESHOLDS:
    name = dictionary_name(theta)
    expanded_df = build_expanded_dictionary_for_threshold(theta)
    expanded_dictionary_map[name] = expanded_df

    out_path = OUTPUT_DIR / f"{name}.csv"
    expanded_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    saved_files.append(out_path)

    counts = (
        expanded_df.groupby(["dimension", "source"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    for source_col in ["seed", "fasttext_candidate"]:
        if source_col not in counts.columns:
            counts[source_col] = 0
    counts["threshold"] = theta
    counts["dictionary_name"] = name
    counts["total_terms"] = counts["seed"] + counts["fasttext_candidate"]
    summary_rows.append(counts[["dictionary_name", "threshold", "dimension", "seed", "fasttext_candidate", "total_terms"]])

summary_df = pd.concat(summary_rows, ignore_index=True)
summary_path = OUTPUT_DIR / "expanded_dictionary_threshold_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
saved_files.append(summary_path)

print("saved files:")
for path in saved_files:
    print("-", path)

display(summary_df)


saved files:
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/seed_dictionary_normalized_for_comparison.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/fasttext_raw_candidates_all_seeds.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_10.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_20.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_30.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_40.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_45.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_50.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_55.csv
- /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_60.csv
- /content/drive/MyDrive/UD_26/fi

source,dictionary_name,threshold,dimension,seed,fasttext_candidate,total_terms
0,expanded_dictionary_theta_0_10,0.10,E,18,6038,6056
1,expanded_dictionary_theta_0_10,0.10,G,17,7252,7269
2,expanded_dictionary_theta_0_10,0.10,S,18,7916,7934
3,expanded_dictionary_theta_0_20,0.20,E,18,6038,6056
4,expanded_dictionary_theta_0_20,0.20,G,17,7252,7269
5,expanded_dictionary_theta_0_20,0.20,S,18,7916,7934
6,expanded_dictionary_theta_0_30,0.30,E,18,6038,6056
7,expanded_dictionary_theta_0_30,0.30,G,17,7252,7269
8,expanded_dictionary_theta_0_30,0.30,S,18,7916,7934
9,expanded_dictionary_theta_0_40,0.40,E,18,4849,4867


In [15]:
candidate_counts = summary_df.pivot_table(
    index="threshold",
    columns="dimension",
    values="fasttext_candidate",
    aggfunc="sum",
    fill_value=0,
).astype(int)

term_counts = summary_df.pivot_table(
    index="threshold",
    columns="dimension",
    values="total_terms",
    aggfunc="sum",
    fill_value=0,
).astype(int)

print("fastText candidate counts by threshold/dimension")
display(candidate_counts)
print("total dictionary terms by threshold/dimension")
display(term_counts)


fastText candidate counts by threshold/dimension


dimension,E,G,S
threshold,,,
0.10,6038,7252,7916
0.20,6038,7252,7916
0.30,6038,7252,7916
0.40,4849,5520,6378
0.45,3056,2021,3366
0.50,1612,1108,1692
0.55,936,668,812
0.60,788,131,186
0.65,289,38,28


total dictionary terms by threshold/dimension


dimension,E,G,S
threshold,,,
0.10,6056,7269,7934
0.20,6056,7269,7934
0.30,6056,7269,7934
0.40,4867,5537,6396
0.45,3074,2038,3384
0.50,1630,1125,1710
0.55,954,685,830
0.60,806,148,204
0.65,307,55,46


In [16]:
review_frames = []
for theta, name in [(theta, dictionary_name(theta)) for theta in THRESHOLDS]:
    candidates = expanded_dictionary_map[name]
    candidates = candidates[candidates["source"].eq("fasttext_candidate")].copy()
    candidates["threshold"] = theta
    review_frames.append(candidates)

manual_review_df = pd.concat(review_frames, ignore_index=True) if review_frames else pd.DataFrame()
manual_review_columns = [
    "threshold", "dimension", "candidate_term", "similarity", "seed_terms_matched",
    "query_terms_matched", "keep_review", "exclude_reason",
]
manual_review_df = manual_review_df[manual_review_columns].sort_values(
    ["threshold", "dimension", "similarity", "candidate_term"],
    ascending=[True, True, True, True],
).reset_index(drop=True)

manual_review_sample = (
    manual_review_df.groupby(["threshold", "dimension"], group_keys=False)
    .head(REVIEW_ROWS_PER_THRESHOLD_DIMENSION)
    .reset_index(drop=True)
)

print("manual_review_df rows:", len(manual_review_df))
print("displaying up to", REVIEW_ROWS_PER_THRESHOLD_DIMENSION, "rows per threshold/dimension")
display(manual_review_sample)


manual_review_df rows: 97143
displaying up to 20 rows per threshold/dimension


,threshold,dimension,candidate_term,similarity,seed_terms_matched,query_terms_matched,keep_review,exclude_reason
0,0.1,E,제조기술,0.346126,재활용,재활용,REVIEW,
1,0.1,E,가전제품의,0.346260,재활용,재활용,REVIEW,
2,0.1,E,파쇄하여,0.346493,재활용,재활용,REVIEW,
3,0.1,E,환경친화적이고,0.346605,재활용,재활용,REVIEW,
4,0.1,E,친환경농산물,0.346788,재활용,재활용,REVIEW,
...,...,...,...,...,...,...,...,...
579,0.7,S,산업재해율,0.738258,산업재해,산업재해,REVIEW,
580,0.7,S,산업재해와,0.745155,산업재해,산업재해,REVIEW,
581,0.7,S,산업재해란,0.746092,산업재해,산업재해,REVIEW,
582,0.7,S,人權,0.752468,인권,인권,REVIEW,
